In [1]:
# installation 
%pip install torch pandas scikit-learn
%pip install mlflow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# importation 

import hashlib
import json
import os
from datetime import datetime
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import FunctionTransformer, RobustScaler

In [3]:
# Fixation de la graine pour la reproductibilité
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# 1. PRÉPROCESSING DES DONNÉES

DERIVEES = ["MCV", "MCH", "MCHC"]
REDONDANTES_LINEAIRE = ["BASOR", "EOSR", "LYMR", "MONOR", "NEUR"]


def nettoyer_donnees(df: pl.DataFrame) -> pl.DataFrame:
    """Remplace les valeurs physiologiquement impossibles par des valeurs manquantes."""
    return df.with_columns(
        HCT=pl.when(pl.col("HCT") <= 1).then(None).otherwise(pl.col("HCT")),
        PLT=pl.when(pl.col("PLT") == 0).then(None).otherwise(pl.col("PLT")),
        WBC=pl.when(pl.col("WBC") == 0).then(None).otherwise(pl.col("WBC")),
        POTASS=pl.when(pl.col("POTASS") > 10).then(None).otherwise(pl.col("POTASS")),
    )


def reparer_sources(df: pl.DataFrame) -> pl.DataFrame:
    """Reconstruit RBC et HCT depuis les indices erythrocytaires."""
    return df.with_columns(
        RBC=pl.coalesce(pl.col("RBC"), pl.col("HGB") / pl.col("MCH") * 10),
        HCT=pl.coalesce(pl.col("HCT"), pl.col("HGB") / pl.col("MCHC") * 100),
    )


def _ratio(num: str, den: str) -> pl.Expr:
    return pl.when(pl.col(den) > 0).then(pl.col(num) / pl.col(den)).otherwise(None)


def ajouter_variables(df: pl.DataFrame) -> pl.DataFrame:
    """Compte de défaillances d'organe et ratios cliniques."""
    return df.with_columns(
        nb_anomalies=pl.sum_horizontal(
            [
                (pl.col("PLT") < 150).cast(pl.Int8),
                (pl.col("CREA") > 1.3).cast(pl.Int8),
                (pl.col("GBIL") > 1.2).cast(pl.Int8),
                (pl.col("NT") < 70).cast(pl.Int8),
                (pl.col("ALB") < 35).cast(pl.Int8),
                (pl.col("LYM") < 1).cast(pl.Int8),
            ]
        ),
        ratio_neu_lym=_ratio("NEU", "LYM"),
        ratio_plt_lym=_ratio("PLT", "LYM"),
        ratio_crp_alb=_ratio("CRP", "ALB"),
        ratio_bun_crea=_ratio("BUN", "CREA"),
    )


def preparer_pipeline(df: pl.DataFrame) -> pl.DataFrame:
    """Enchaîne le nettoyage, la réparation et l'ingénierie de variables."""
    return ajouter_variables(reparer_sources(nettoyer_donnees(df)).drop(DERIVEES))


def build_scikit_preprocessor(X_train: pl.DataFrame):
    """Construit le ColumnTransformer sklearn d'imputation et de mise à l'échelle."""
    cols = [c for c in X_train.columns if c not in REDONDANTES_LINEAIRE]
    a_log = [c for c in cols if not c.startswith("nb_") and (X_train[c].skew() or 0) > 1]
    autres = [c for c in cols if c not in a_log]

    log_pipeline = ColumnTransformer(
        transformers=[("identity", FunctionTransformer(np.log1p), a_log)]
    )

    log_prep = nn.Sequential()  # Utilisé comme placeholder conceptuel

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "asymetriques",
                nn.Sequential() if False else None,  # Pipeline sklearn
                a_log,
            ),
        ],
        remainder="drop",
    )

    # Définition formelle des sous-pipelines Sklearn
    from sklearn.pipeline import Pipeline

    log_pipe = Pipeline(
        [
            ("log", FunctionTransformer(np.log1p)),
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", RobustScaler()),
        ]
    )

    autres_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", RobustScaler()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("asymetriques", log_pipe, a_log),
            ("autres", autres_pipe, autres),
        ],
        verbose_feature_names_out=False,
    )
    return preprocessor

In [5]:
# 2. ARCHITECTURE DU MODÈLE PYTORCH 

class TabularResBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        return x + self.block(x)


class TabularPyTorchModel(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128, num_blocks: int = 2, dropout: float = 0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )
        self.blocks = nn.ModuleList(
            [TabularResBlock(hidden_dim, dropout) for _ in range(num_blocks)]
        )
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(-1)

In [6]:
# 3. ENTRAÎNEMENT & OPTIMISATION DE SEUIL


def trouver_seuil_optimal(y_true: np.ndarray, y_probs: np.ndarray):
    """Recherche fine du seuil qui maximise le F1-Score."""
    best_th = 0.5
    best_f1 = 0.0
    thresholds = np.linspace(0.05, 0.95, 181)

    for th in thresholds:
        preds = (y_probs >= th).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_th = th

    return best_th, best_f1


def train_pytorch_pipeline(data_path: str, n_splits: int = 5, epochs: int = 60, batch_size: int = 128):
    # Chargement
    df_raw = pl.read_csv(data_path, null_values="NA")
    df_raw = df_raw.with_columns(
        pl.col("BloodCulture").replace_strict({"no": 0, "yes": 1}).cast(pl.Int8)
    )

    # Préparation des features
    df_prep = preparer_pipeline(df_raw)
    X = df_prep.select(pl.exclude("ID", "BloodCulture"))
    y = df_prep["BloodCulture"].to_numpy()

    X_pandas = X.to_pandas()
    input_dim = None
    oof_probs = np.zeros(len(y))

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    models = []
    preprocessors = []  # On conserve le preprocessor de chaque fold

    print(f"--- Début de l'entraînement PyTorch sans fuite de données ({n_splits}-Fold CV) ---")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_pandas, y)):
        # 1. Séparation stricte des données AVANT le prétraitement
        X_tr_raw, X_va_raw = X_pandas.iloc[train_idx], X_pandas.iloc[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]

        # 2. Fit du preprocessor UNIQUEMENT sur le train du fold courant
        fold_preprocessor = build_scikit_preprocessor(pl.from_pandas(X_tr_raw))
        X_tr_trans = fold_preprocessor.fit_transform(X_tr_raw)
        X_va_trans = fold_preprocessor.transform(X_va_raw)  # Simple transform !

        preprocessors.append(fold_preprocessor)
        input_dim = X_tr_trans.shape[1]

        # Convertir en tenseurs PyTorch
        X_tr, y_tr_t = torch.tensor(X_tr_trans, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32)
        X_va, y_va_t = torch.tensor(X_va_trans, dtype=torch.float32), torch.tensor(y_va, dtype=torch.float32)

        train_ds = TensorDataset(X_tr, y_tr_t)
        val_ds = TensorDataset(X_va, y_va_t)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        # Calcul du poids pour gérer l'imbalance (~8 % de positifs)
        pos_weight = torch.tensor([(len(y_tr) - y_tr.sum()) / y_tr.sum()]).to(DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        model = TabularPyTorchModel(input_dim=input_dim).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

        best_val_f1 = 0.0
        best_model_weights = None

        for epoch in range(epochs):
            model.train()
            for bx, by in train_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                optimizer.zero_grad()
                logits = model(bx)
                loss = criterion(logits, by)
                loss.backward()
                optimizer.step()

            # Validation
            model.eval()
            val_logits = []
            with torch.no_grad():
                for bx, _ in val_loader:
                    bx = bx.to(DEVICE)
                    val_logits.append(model(bx))
            
            val_logits = torch.cat(val_logits)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()

            _, current_f1 = trouver_seuil_optimal(y_va, val_probs)
            scheduler.step(current_f1)

            if current_f1 > best_val_f1:
                best_val_f1 = current_f1
                best_model_weights = model.state_dict().copy()

        model.load_state_dict(best_model_weights)
        model.eval()

        # Probas Out-Of-Fold
        with torch.no_grad():
            oof_logits = torch.cat([model(bx.to(DEVICE)) for bx, _ in val_loader])
            oof_probs[val_idx] = torch.sigmoid(oof_logits).cpu().numpy()

        models.append(model)
        print(f"Fold {fold+1}/{n_splits} - Meilleur F1 Val: {best_val_f1:.4f}")

    # Optimisation globale du seuil sur les prédictions Out-Of-Fold (OOF)
    best_threshold, overall_f1 = trouver_seuil_optimal(y, oof_probs)
    preds_binary = (oof_probs >= best_threshold).astype(int)

    print("\n================ RÉSULTATS CV ================")
    print(f"Seuil optimal global : {best_threshold:.4f}")
    print(f"F1 Score OOF         : {overall_f1:.4f}")
    print(f"Précision OOF        : {precision_score(y, preds_binary):.4f}")
    print(f"Rappel OOF           : {recall_score(y, preds_binary):.4f}")
    print(f"ROC AUC OOF          : {roc_auc_score(y, oof_probs):.4f}")
    print("==============================================")

    return models, preprocessors, best_threshold

In [7]:
# 4. PRÉDICTION SUR LE JEU DE TEST


def predire_jeu_de_test(
    test_csv_path: str,
    models: list,
    preprocessors: list,
    threshold: float,
    output_filename: str = "Groupe5_ADALR.csv",
):
    df_test_raw = pl.read_csv(test_csv_path, null_values="NA")
    ids_test = df_test_raw["ID"].to_numpy()

    df_test_prep = preparer_pipeline(df_test_raw)
    X_test = df_test_prep.select(pl.exclude("ID", "BloodCulture")).to_pandas()

    test_probs = np.zeros(len(df_test_raw))

    # Chaque modèle utilise le préprocesseur ajusté sur son propre fold
    for model, prep in zip(models, preprocessors):
        X_test_trans = prep.transform(X_test)
        X_test_tensor = torch.tensor(X_test_trans, dtype=torch.float32).to(DEVICE)
        
        model.eval()
        with torch.no_grad():
            logits = model(X_test_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
            test_probs += probs / len(models)

    preds_test = (test_probs >= threshold).astype(int)

    submission = pd.DataFrame({"ID": ids_test, "pred": preds_test})
    submission.to_csv(output_filename, index=False)
    print(f"\nFichier de prédiction généré avec succès : {output_filename}")

In [8]:
if __name__ == "__main__":
    # 1. Chemin vers votre fichier d'ENTRAÎNEMENT 
    DATA_PATH = r"C:/Users/ELITE COMPUTER/OneDrive - etu.unistra.fr/Bureau/projet 1 grp 5/bacteriemie.csv"
    # Note : Si votre fichier est compressé, mettez : "bacteriemie.csv.gz"

    # 2. Lancement de l'entraînement et recherche du seuil optimal
    models, preprocessor, best_th = train_pytorch_pipeline(
        data_path=DATA_PATH, 
        n_splits=5, 
        epochs=50
    )

    # 3. Prédictions sur le jeu de TEST
    TEST_PATH = r"C:/Users/ELITE COMPUTER/OneDrive - etu.unistra.fr/Bureau/projet 1 grp 5/bacteriemie_test.csv"
    
    predire_jeu_de_test(
        test_csv_path=TEST_PATH,
        models=models,
        preprocessor=preprocessor,
        threshold=best_th,
        output_filename="Groupe5_ADALR.csv" 
    )

--- Début de l'entraînement PyTorch sans fuite de données (5-Fold CV) ---
Fold 1/5 - Meilleur F1 Val: 0.3747
Fold 2/5 - Meilleur F1 Val: 0.3389
Fold 3/5 - Meilleur F1 Val: 0.3881
Fold 4/5 - Meilleur F1 Val: 0.3319
Fold 5/5 - Meilleur F1 Val: 0.3548

================ RÉSULTATS CV ================
Seuil optimal global : 0.5350
F1 Score OOF         : 0.2907
Précision OOF        : 0.2161
Rappel OOF           : 0.4439
ROC AUC OOF          : 0.7323


TypeError: predire_jeu_de_test() got an unexpected keyword argument 'preprocessor'. Did you mean 'preprocessors'?